In [80]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms
import statsmodels.api as sm
from statsmodels.compat import lzip
import matplotlib.pyplot as plt
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.diagnostic import het_arch

!pip install arch
import pickle
from arch.unitroot import DFGLS, ADF, KPSS, PhillipsPerron
from arch import arch_model

In [59]:
data_sp = pd.read_csv('S&P 500 Data.csv')
data_bit = pd.read_excel('Bitcoin.xlsx')

#data_sp = pickle.load(pd.read_csv(r"S&P 500 Data.csv"))

In [60]:
print(data_sp.head())
print(data_bit.head())
data_sp.drop("Vol.", axis=1, inplace=True)

price_cols = ["Price", "Open", "High", "Low"]

price_cols = ["Price", "Open", "High", "Low"]

for col in price_cols:
    # Convertir en string d'abord
    data_sp[col] = data_sp[col].astype(str)
    
    # Supprimer les virgules
    data_sp[col] = data_sp[col].str.replace(",", "", regex=False)
    
    # Convertir en float
    data_sp[col] = pd.to_numeric(data_sp[col], errors="coerce")
print(data_sp.head())

         Date     Price      Open      High       Low  Vol. Change %
0  12/29/2025  6,904.01  6,901.27  6,920.02  6,897.62   NaN   -0.37%
1  12/26/2025  6,929.94  6,936.02  6,945.77  6,921.60   NaN   -0.03%
2  12/24/2025  6,932.05  6,904.91  6,937.32  6,904.91   NaN    0.32%
3  12/23/2025  6,909.79  6,872.41  6,910.88  6,868.81   NaN    0.46%
4  12/22/2025  6,878.49  6,865.21  6,882.03  6,855.74   NaN    0.64%
                   timeOpen                 timeClose  \
0  2025-12-28T00:00:00.000Z  2025-12-28T23:59:59.999Z   
1  2025-12-27T00:00:00.000Z  2025-12-27T23:59:59.999Z   
2  2025-12-26T00:00:00.000Z  2025-12-26T23:59:59.999Z   
3  2025-12-25T00:00:00.000Z  2025-12-25T23:59:59.999Z   
4  2025-12-24T00:00:00.000Z  2025-12-24T23:59:59.999Z   

                   timeHigh                   timeLow  name          open  \
0  2025-12-28T14:43:00.000Z  2025-12-28T20:07:00.000Z  2781  87799.345179   
1  2025-12-27T23:52:00.000Z  2025-12-27T00:23:00.000Z  2781  87301.433186   
2  2025-12-2

## Augmented Dickey-Fuller (ADF) unit-root test.

In [61]:
##### Compute the log return of the bitcoin price
data_bit["log_return"] = np.log(data_bit["close"] / data_bit["close"].shift(1))
      # ou  data_bit["log_return"] = np.log(data_bit["close"]).diff()

# Compute the log return of the S&P 500
data_sp["log_return"] = np.log(data_sp["Price"] / data_sp["Price"].shift(1))
      # ou data_sp["log_return"] = np.log(data_sp["Price"]).diff()

In [62]:
# Test for Bitcoin
data_bit = data_bit.dropna()
print(data_bit.head())
res_bit = ADF(data_bit["log_return"], lags=10)

data_sp = data_sp.dropna()
print(data_sp.head())
res_sp = ADF(data_sp["log_return"], lags=10)

                   timeOpen                 timeClose  \
1  2025-12-27T00:00:00.000Z  2025-12-27T23:59:59.999Z   
2  2025-12-26T00:00:00.000Z  2025-12-26T23:59:59.999Z   
3  2025-12-25T00:00:00.000Z  2025-12-25T23:59:59.999Z   
4  2025-12-24T00:00:00.000Z  2025-12-24T23:59:59.999Z   
5  2025-12-23T00:00:00.000Z  2025-12-23T23:59:59.999Z   

                   timeHigh                   timeLow  name          open  \
1  2025-12-27T23:52:00.000Z  2025-12-27T00:23:00.000Z  2781  87301.433186   
2  2025-12-26T07:26:00.000Z  2025-12-26T15:17:00.000Z  2781  87235.511696   
3  2025-12-25T15:46:00.000Z  2025-12-25T23:47:00.000Z  2781  87608.316603   
4  2025-12-24T23:01:00.000Z  2025-12-24T14:40:00.000Z  2781  87404.321786   
5  2025-12-23T00:40:00.000Z  2025-12-23T14:58:00.000Z  2781  88490.028305   

           high           low         close        volume     marketCap  \
1  87874.782544  87182.979315  87802.155212  1.374120e+10  1.753178e+12   
2  89459.432726  86628.144459  87301.432240 

In [63]:
print(res_bit.summary())
print(res_sp.summary())

   Augmented Dickey-Fuller Results   
Test Statistic                -20.356
P-value                         0.000
Lags                               10
-------------------------------------

Trend: Constant
Critical Values: -3.43 (1%), -2.86 (5%), -2.57 (10%)
Null Hypothesis: The process contains a unit root.
Alternative Hypothesis: The process is weakly stationary.
   Augmented Dickey-Fuller Results   
Test Statistic                -19.052
P-value                         0.000
Lags                               10
-------------------------------------

Trend: Constant
Critical Values: -3.43 (1%), -2.86 (5%), -2.57 (10%)
Null Hypothesis: The process contains a unit root.
Alternative Hypothesis: The process is weakly stationary.


In [65]:
pp_bit = PhillipsPerron(data_bit["log_return"], trend="ct")

pp_sp = PhillipsPerron(data_sp["log_return"], trend="ct")

print(pp_bit.summary())
print(pp_sp.summary())

     Phillips-Perron Test (Z-tau)    
Test Statistic                -80.454
P-value                         0.000
Lags                               33
-------------------------------------

Trend: Constant and Linear Time Trend
Critical Values: -3.96 (1%), -3.41 (5%), -3.13 (10%)
Null Hypothesis: The process contains a unit root.
Alternative Hypothesis: The process is weakly stationary.
     Phillips-Perron Test (Z-tau)    
Test Statistic                -71.011
P-value                         0.000
Lags                               31
-------------------------------------

Trend: Constant and Linear Time Trend
Critical Values: -3.96 (1%), -3.41 (5%), -3.13 (10%)
Null Hypothesis: The process contains a unit root.
Alternative Hypothesis: The process is weakly stationary.


### Tests de normalité de Jacque-Bera

In [68]:
# 
print
x_bit = data_bit["log_return"]

desc_bit = {
    "Mean": x_bit.mean(),
    "Std": x_bit.std(),
    "Skewness": x_bit.skew(),
    "Kurtosis": x_bit.kurt(),
    "Min": x_bit.min(),
    "Max": x_bit.max(),
    "JB": jarque_bera(x_bit)[0],
    "JB p-value": jarque_bera(x_bit)[1]
}

desc_bit

desc_bit_table = pd.DataFrame(desc_bit, index=["log_return"])
print(desc_bit_table)

#
x_sp = data_sp["log_return"]

desc_sp = {
    "Mean": x_sp.mean(),
    "Std": x_sp.std(),
    "Skewness": x_sp.skew(),
    "Kurtosis": x_sp.kurt(),
    "Min": x_sp.min(),
    "Max": x_sp.max(),
    "JB": jarque_bera(x_sp)[0],
    "JB p-value": jarque_bera(x_sp)[1]
}

desc_sp

desc_sp_table = pd.DataFrame(desc_sp, index=["log_return"])

print(desc_sp_table)



#jb_stat_sp, jb_pvalue_sp, skew_sp, kurtosis_sp = jarque_bera(data_sp["log_return"])

#jb_stat_bit, jb_pvalue_bit, skew_bit, kurtosis_bit = jarque_bera(data_bit["log_return"])"""


               Mean       Std  Skewness   Kurtosis       Min       Max  \
log_return -0.00252  0.047543  1.024638  22.664772 -0.395243  0.675195   

                       JB  JB p-value  
log_return  121629.796349         0.0  
                Mean       Std  Skewness   Kurtosis       Min       Max  \
log_return -0.000476  0.010911  0.626888  14.009304 -0.090895  0.127657   

                      JB  JB p-value  
log_return  32215.521612         0.0  


### Test de l'hétéroscédasticité conditionnelle

In [78]:

data1_bit = sm.add_constant(data_bit["log_return"])
results_bit = sm.OLS(data1_bit["log_return"], data1_bit['const']).fit()
print(results_bit.summary())

data1_sp = sm.add_constant(data_sp["log_return"])
results_sp = sm.OLS(data1_sp["log_return"], data1_sp['const']).fit()
print(results_sp.summary())


archres_bit = het_arch(results_bit.resid, maxlag=5)
name_bit = ['lm_bit', 'lmpval_bit', 'fval_bit', 'fpval_bit']
print(lzip(name_bit, archres_bit))

archres_sp = het_arch(results_sp.resid, maxlag=5)
name_sp = ['lm_sp', 'lmpval_sp', 'fval_sp', 'fpval_sp']
print(lzip(name_sp, archres_sp))

                            OLS Regression Results                            
Dep. Variable:             log_return   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                       nan
Date:                Fri, 16 Jan 2026   Prob (F-statistic):                nan
Time:                        12:54:39   Log-Likelihood:                 9189.2
No. Observations:                5647   AIC:                        -1.838e+04
Df Residuals:                    5646   BIC:                        -1.837e+04
Df Model:                           0                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0025      0.001     -3.982      0.0

C:\Users\emman\AppData\Local\Temp\ipykernel_1856\2841320538.py:10: FutureWarning: the 'maxlag' keyword is deprecated, use 'nlags' instead.
  archres_bit = het_arch(results_bit.resid, maxlag=5)
C:\Users\emman\AppData\Local\Temp\ipykernel_1856\2841320538.py:14: FutureWarning: the 'maxlag' keyword is deprecated, use 'nlags' instead.
  archres_sp = het_arch(results_sp.resid, maxlag=5)


### GARCH estimation

In [84]:
data_bit["log_return_scaled"] = data_bit["log_return"] * 100  # passer à % plutôt qu'à fraction


am_bit = arch_model(data_bit["log_return_scaled"], vol='GARCH')
resgar_bit = am_bit.fit()
print(resgar_bit.summary())

Iteration:      1,   Func. Count:      6,   Neg. LLF: 37704.22162155704
Iteration:      2,   Func. Count:     14,   Neg. LLF: 2258822.890542565
Iteration:      3,   Func. Count:     21,   Neg. LLF: 15404.736681423123
Iteration:      4,   Func. Count:     27,   Neg. LLF: 15418.912425806024
Iteration:      5,   Func. Count:     33,   Neg. LLF: 15273.493001020624
Iteration:      6,   Func. Count:     38,   Neg. LLF: 15271.434494273368
Iteration:      7,   Func. Count:     44,   Neg. LLF: 15267.854609267548
Iteration:      8,   Func. Count:     49,   Neg. LLF: 15267.84970196995
Iteration:      9,   Func. Count:     54,   Neg. LLF: 15267.849367888946
Iteration:     10,   Func. Count:     59,   Neg. LLF: 15267.849294859121
Iteration:     11,   Func. Count:     64,   Neg. LLF: 15267.849315048286
Optimization terminated successfully    (Exit mode 0)
            Current function value: 15267.849294891485
            Iterations: 11
            Function evaluations: 74
            Gradient evalua

In [87]:
!pip install --upgrade arch
!pip install rpy2 numpy pandas arch



   ---------- ----------------------------- 1/4 [rpy2-rinterface]
   -------------------- ------------------- 2/4 [rpy2-robjects]
   -------------------- ------------------- 2/4 [rpy2-robjects]
   ------------------------------ --------- 3/4 [rpy2]
   ---------------------------------------- 4/4 [rpy2]



NameError: name 'install' is not defined

In [88]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

pandas2ri.activate()
rugarch = importr('rugarch')

# Préparer données
returns = data_bit["log_return_scaled"].dropna()
with ro.conversion.localconverter(ro.default_converter + pandas2ri.converter):
    r_data = ro.conversion.py2rpy(returns)

models = {
    'AR-GARCH': 'sGARCH',
    'AR-EGARCH': 'eGARCH', 
    'AR-TGARCH': 'gjrGARCH',
    'AR-APARCH': 'apARCH',
    'AR-CGARCH': 'csGARCH',
    'AR-ACGARCH': 'fGARCH'
}

results = {}

for name, r_model in models.items():
    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")
    
    try:
        # Spécification
        spec = rugarch.ugarchspec(
            mean_model=ro.ListVector({
                'armaOrder': ro.IntVector([1, 0]),
                'include.mean': True
            }),
            variance_model=ro.ListVector({
                'model': r_model,
                'garchOrder': ro.IntVector([1, 1])
            }),
            distribution_model='sstd'
        )
        
        # Estimation
        fit = rugarch.ugarchfit(spec=spec, data=r_data)
        
        # Afficher le summary R complet
        rugarch.show(fit)
        
        # Stocker
        results[name] = fit
        
    except Exception as e:
        print(f"ERREUR: {str(e)}")

# Résumé final comparatif
print("\n" + "="*80)
print("RÉSUMÉ COMPARATIF")
print("="*80)

for name, fit in results.items():
    try:
        infocrit = rugarch.infocriteria(fit)
        print(f"\n{name}:")
        print(f"  AIC: {infocrit[0]:.2f}, BIC: {infocrit[1]:.2f}, LLH: {rugarch.likelihood(fit)[0]:.2f}")
    except:
        pass


"""
models = ['GARCH', 'EGARCH', 'GJR', 'APARCH', 'CGARCH', 'ACGARCH']
results_bit = {}

for m in models:
    am = arch_model(
        data_bit["log_return_scaled"],
        mean='AR',
        lags=1,        # AR(1), changer si nécessaire
        vol=m,
        p=1, q=1
    )
    res = am.fit(disp='off')
    results_bit[m] = res
    print(f"--- {m} ---")
    print(res.summary())


models = ['GARCH', 'EGARCH', 'GJR', 'APARCH', 'CGARCH', 'ACGARCH']
results = {}

for m in models:
    am = arch_model(
        data_bit["log_return"],
        mean='AR',
        lags=1,        # AR(1), changer si nécessaire
        vol=m,
        p=1, q=1
    )
    res = am.fit(disp='off')
    results[m] = res
    print(f"--- {m} ---")
    print(res.summary())

"""

TypeError: 'NoneType' object is not iterable